In [ ]:
import os
print("Current folder:", os.getcwd())
print("Files here:", os.listdir())

In [ ]:
#Part 1: Load the Data
from pyspark.sql import SparkSession
print("PySpark import works")

In [ ]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — ready")

In [ ]:
print("Files in current folder:", os.listdir())

if "data" in os.listdir():
    print("Files in data folder:", os.listdir("data"))

In [ ]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Record count: {df.count()}")
df.printSchema()
df.show(10, truncate=False)

In [ ]:
#Part 2: Basic Aggregations
#Task 2.1 — Transaction count and total revenue per store
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp should now be 'timestamp (nullable = true)'

In [ ]:
#Task 2.2 — Stats per category

from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
        _round(avg("amount"), 2).alias("avg_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

In [ ]:
#task 2.2
from pyspark.sql.functions import count, sum as _sum, min as _min, max as _max, round as _round

df.groupBy("category") \
  .agg(
      count("tx_id").alias("tx_count"),
      _round(_sum("amount"), 2).alias("total_PLN"),
      _round(_min("amount"), 2).alias("min_PLN"),
      _round(_max("amount"), 2).alias("max_PLN")
  ) \
  .orderBy("category") \
  .show()

In [ ]:
#Task 3.1 — Transaction count per hour (tumbling 1h)

from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # 1-hour tumbling window
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

In [ ]:
(
    hourly
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN",
    )
    .show(truncate=False)
)

In [ ]:
#Task 3.2:  30-minute windows per store 
from pyspark.sql.functions import window, count, sum as _sum, round as _round, col

df.groupBy(window("timestamp", "30 minutes"), "store") \
  .agg(
      count("tx_id").alias("tx_count"),
      _round(_sum("amount"), 2).alias("total_PLN")
  ) \
  .select(
      col("window.start").alias("from"),
      col("window.end").alias("to"),
      "store",
      "tx_count",
      "total_PLN"
  ) \
  .orderBy("from", "store") \
  .show(truncate=False)

In [ ]:
#Task 3.3:  Which hour had the highest revenue for store “Kraków”?
from pyspark.sql.functions import desc, window, count, sum as _sum, round as _round, col

df.filter(col("store") == "Kraków") \
  .groupBy(window("timestamp", "1 hour")) \
  .agg(
      count("tx_id").alias("tx_count"),
      _round(_sum("amount"), 2).alias("total_PLN")
  ) \
  .select(
      col("window.start").alias("from"),
      col("window.end").alias("to"),
      "tx_count",
      "total_PLN"
  ) \
  .orderBy(desc("total_PLN")) \
  .show(truncate=False)

In [ ]:
#Part 4: Sliding Windows

sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # 1h width, 30min step
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN",
    )
    .orderBy("from")
)
sliding.show(truncate=False)

In [ ]:
#Task 4.2 — Compare tumbling vs sliding
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):         {tumbling_rows} windows")
print(f"Sliding  (1h / 30min): {sliding_rows} windows")

# Answer in a comment: why does sliding produce more rows?
#  ANSWER:
## Sliding produces more rows because the windows overlap.
## A new 1-hour window starts every 30 minutes, so Spark creates more time intervals than in tumbling windows, where windows are back-to-back and do not overlap.

In [ ]:
# Answer in comments:

# 1. How many transactions are in the 09:00–10:00 window?
#    Check the result from Task 3.1.
#    ANSWER: 4661: |2026-04-12 09:00:00|2026-04-12 10:00:00|4661    |1896230.21|

# 2. What is the difference between groupBy("store") and groupBy(window(...), "store")?
#    ANSWER:# groupBy("store") aggregates all transactions for each store across the whole dataset.
# groupBy(window(...), "store") aggregates transactions separately for each store inside each time window.

# 3. In the sliding window (1h / 30min), how many windows contain transactions from 09:30?
#    Hint: draw a timeline.
#    ANSWER: 3 windows contain transactions from 09:30: 08:30–09:30, 09:00–10:00, and 09:30–10:30.
## 2026-04-12 08:30:00|2026-04-12 09:30:00|4443    |1753033.6 |
## |2026-04-12 09:00:00|2026-04-12 10:00:00|4661    |1896230.21|
## |2026-04-12 09:30:00|2026-04-12 10:30:00|3696    |1557641.39|

In [ ]:
#Homework
#1.1 Calculate the average transaction amount and find out the hour when the dtore has lowest transactions

from pyspark.sql.functions import avg, round as _round, col, window

gdansk_avg = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("avg_PLN")
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "avg_PLN"
    )
    .orderBy("avg_PLN")  
)

gdansk_avg.show(truncate=False)

#Answer: 2026-04-12 08:00:00|2026-04-12 09:00:00|395.01 PLN
## 2026-04-12 10:00:00|2026-04-12 11:00:00|412.92 PLN
## |2026-04-12 09:00:00|2026-04-12 10:00:00|415.91 |


In [ ]:
#30-minute windows per store
from pyspark.sql.functions import avg, round as _round, col, window

gdansk_avg_30min = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "30 minutes"))  
    .agg(
        _round(avg("amount"), 2).alias("avg_PLN")
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "avg_PLN"
    )
    .orderBy("avg_PLN")   # lowest first
)

gdansk_avg_30min.show(truncate=False)

#Answer: From 8:00-9:00 store Gdańsk had the lowest average transaction amount.

In [ ]:
#2. Count how many transactions per category occurred in the 09:00–09:30 window.

from pyspark.sql.functions import count

window_0900_0930 = (
    df.filter(
        (col("timestamp") >= "2026-04-12 09:00:00") &
        (col("timestamp") <  "2026-04-12 09:30:00")
    )
    .groupBy("category")
    .agg(count("tx_id").alias("tx_count"))
    .orderBy("category")
)

window_0900_0930.show()

#Answer: elektronika: 611, książki: 622, odzież: 605, żywność:  567

In [ ]:
#3. Use a 15-minute window and find which quarter-hour had the peak transaction volume (all stores combined).

from pyspark.sql.functions import desc

quarter_peak = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("tx_count"))
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count"
    )
    .orderBy(desc("tx_count"))   #show highest first
)

quarter_peak.show(truncate=False)

#Answer: From 9:15 - 9:30, there is highest number of transactions in all stores

In [ ]:
spark.stop()